# 🏠 RoomCanvas AI — Inference Server

**Architecture:** Realistic Vision V6.0 + MultiControlNet (MLSD + MiDaS Depth) + DPM++ 2M Karras + sd-vae-ft-mse  
**Purpose:** Persistent HTTP inference server inside a Colab T4 GPU runtime, exposed via ngrok.  
**Endpoint:** `POST /infer` accepts a room image + style key, returns 3 redesigned variations + metadata.

---
### Key upgrades vs. original SD1.5 + single MLSD ControlNet
- **MiDaS depth ControlNet** alongside MLSD — grounds furniture, eliminates floating objects
- **Realistic Vision V6.0** checkpoint — stronger photorealism than vanilla SD1.5
- **sd-vae-ft-mse** VAE — better color saturation and fine detail
- **DPM++ 2M Karras** at 22 steps — equal/better quality, partially offsets dual-ControlNet overhead

---
### Quick Start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells **1 → 20** top to bottom (Cell 6 will ask you to mount Drive — do it)
3. Paste your free **ngrok auth token** into Cell 5's `NGROK_AUTH_TOKEN` field
4. After Cell 20 finishes, **run Cell 21** to test the live server

> ⚠️ **Do not run a training loop from this notebook.** Inference-only using pretrained weights.


## Section 2 — Environment & GPU Check

This cell **must pass** before anything else. If it raises `RuntimeError`, go to **Runtime → Change runtime type → T4 GPU**.


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then reconnect.'
    )
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'torch.cuda.is_available() returned False even though nvidia-smi passed. '
        'Try restarting the runtime.'
    )

print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'✅ VRAM: {total_vram:.1f} GB total')
print(f'✅ PyTorch version: {torch.__version__}')


## Section 3 — Dependency Installation

**Strategy:** We do NOT touch `torch` or `torchvision`. Colab pre-installs a version compiled together and ABI-matched.
Overwriting either breaks native C++ ops (e.g. `torchvision::nms`).
`xformers` is left unpinned so pip selects the wheel for the installed torch.
All other packages are pinned. **This cell takes ~2-4 minutes on first run.**


In [ ]:
import subprocess, sys

# Do NOT add torch/torchvision/pillow/numpy -- Colab pre-installs ABI-matched versions.
# Reinstalling any of them causes mixed-version crashes.
# xformers left unpinned so pip picks the right wheel.

packages = [
    'diffusers==0.31.0',
    'transformers==4.46.1',
    'accelerate==1.0.1',
    'controlnet-aux==0.0.9',
    'xformers',
    'fastapi==0.115.2',
    'uvicorn==0.34.0',
    'pyngrok==7.2.0',
    'python-multipart==0.0.18',
    'nest-asyncio==1.6.0',
    'safetensors>=0.4.5',   # Realistic Vision ships as .safetensors -- required
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError('pip install failed -- see stderr above.')

print('Installation complete.')

import torch, torchvision
from torchvision.ops import nms as _nms_check
from PIL import Image as _pil_check
import numpy as _np_check
print(f'✅ torch=={torch.__version__}  torchvision=={torchvision.__version__}')
print(f'✅ pillow=={_pil_check.__version__}  numpy=={_np_check.__version__}')
print('✅ C++ ops (torchvision::nms) intact')


In [ ]:
import diffusers, transformers, accelerate, fastapi, uvicorn, pyngrok, PIL, numpy, nest_asyncio
import importlib.metadata

def get_ver(mod, name):
    v = getattr(mod, '__version__', None)
    if v is None:
        try:
            v = importlib.metadata.version(name)
        except Exception:
            v = 'unknown'
    return v

versions = {
    'diffusers':    (get_ver(diffusers,    'diffusers'),    '0.31.0'),
    'transformers': (get_ver(transformers, 'transformers'), '4.46.1'),
    'accelerate':   (get_ver(accelerate,   'accelerate'),   '1.0.1'),
    'fastapi':      (get_ver(fastapi,      'fastapi'),      '0.115.2'),
    'uvicorn':      (get_ver(uvicorn,      'uvicorn'),      '0.34.0'),
    'pyngrok':      (get_ver(pyngrok,      'pyngrok'),      '7.2.0'),
    'pillow':       (get_ver(PIL,          'pillow'),        None),
    'numpy':        (get_ver(numpy,        'numpy'),         None),
    'nest_asyncio': (get_ver(nest_asyncio, 'nest-asyncio'), None),
}

all_ok = True
for pkg, (installed, expected) in versions.items():
    status = '✅'
    if expected and installed != expected:
        status = f'⚠️  MISMATCH (expected {expected})'
        all_ok = False
    print(f'{status}  {pkg}=={installed}')

try:
    xf_ver = importlib.metadata.version('xformers')
    print(f'✅  xformers=={xf_ver}  (auto-selected for installed torch)')
except Exception:
    print('⚠️  xformers not found -- memory-efficient attention falls back to standard.')

print('\n✅ All packages verified.' if all_ok else '\n⚠️ Version mismatches above.')


## Section 4 — Imports


In [ ]:
import io
import base64
import time
import logging
import os
import asyncio
import threading

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    AutoencoderKL,
    DPMSolverMultistepScheduler,
)
from transformers import CLIPProcessor, CLIPModel
from controlnet_aux import MLSDdetector, MidasDetector

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
import uvicorn
import nest_asyncio
import requests
from pyngrok import ngrok

print('✅ All imports successful.')


## Section 5 — Configuration

**All constants live here.** Nothing is hardcoded anywhere else in the notebook.  
Get a free ngrok token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).


In [ ]:
IMG_SIZE = (512, 512)

# DPM++ Karras converges in 22 steps; was 30 with PNDM (sec 5.2)
NUM_INFERENCE_STEPS = 22

# Geometry-preservation tuning
STRENGTH       = 0.65   # 0.60 was too restrictive; 0.65 gives model room to render texture
GUIDANCE_SCALE = 7.5    # better prompt adherence and surface texture rendering

# MultiControlNet weights defined in generate_variations() as [1.0, 0.55]
# (MLSD dominant for geometry, depth secondary for grounding). Tune there, not here.

SEEDS = [42, 123, 777]  # MUST NOT CHANGE -- backend history/regenerate depends on these

ROOM_LABELS = ['bedroom', 'living_room', 'kitchen', 'office', 'dining_room']

CLIP_LABEL_PROMPTS = {
    'bedroom': [
        'a photo of a bedroom',
        'a bedroom interior with a bed, nightstands, and pillows',
        'a sleeping room with bed furniture and soft lighting',
        'interior design photo of a bedroom',
    ],
    'living_room': [
        'a photo of a living room',
        'a living room with a sofa, couch, coffee table, and television',
        'a lounge or sitting room with upholstered furniture',
        'interior design photo of a living room',
    ],
    'kitchen': [
        'a photo of a kitchen',
        'a kitchen with cabinets, countertops, and appliances',
        'a cooking area with an oven, stove, and sink',
        'interior design photo of a modern kitchen',
    ],
    'office': [
        'a photo of an office',
        'a home office with a desk, chair, and computer monitor',
        'a workspace or study room with shelves and books',
        'interior design photo of a home office',
    ],
    'dining_room': [
        'a photo of a dining room',
        'a dining room with a table, chairs, and chandelier',
        'a formal eating area with dining furniture',
        'interior design photo of a dining room',
    ],
}

DRIVE_CACHE_DIR  = '/content/drive/MyDrive/roomcanvas_model_cache'
NGROK_AUTH_TOKEN = '' #@param {type:"string"}

print('✅ Configuration loaded.')
print(f'   Seeds: {SEEDS}  |  IMG_SIZE: {IMG_SIZE}  |  Steps: {NUM_INFERENCE_STEPS}')
print(f'   Geometry: strength={STRENGTH}  guidance={GUIDANCE_SCALE}  controlnet_scales=[1.0, 0.55]')


## Section 6 — Model Download & Weight Caching (Google Drive)

Mounts Drive and points diffusers at a persistent cache. **First run:** ~8 GB download. **Subsequent runs:** ~30s from cache.  
If you skip Drive mounting, weights download to `/content` — works but does NOT persist across disconnections.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    cache_dir = DRIVE_CACHE_DIR
    print(f'✅ Drive mounted. Model cache: {cache_dir}')
except Exception as e:
    cache_dir = '/content/model_cache'
    os.makedirs(cache_dir, exist_ok=True)
    print(f'⚠️  Drive mount failed or skipped: {e}')
    print(f'   Using local cache (will not persist): {cache_dir}')


## Section 7 — Model Loading & Pipeline Initialization

Loads VAE, both ControlNets (MLSD + Depth), then the Realistic Vision V6.0 pipeline.
First run downloads ~8 GB to Drive cache; subsequent runs load in ~30s.


In [ ]:
print('Loading VAE (stabilityai/sd-vae-ft-mse)...')
try:
    vae = AutoencoderKL.from_pretrained(
        'stabilityai/sd-vae-ft-mse',
        torch_dtype=torch.float16,
        cache_dir=cache_dir,
    )
except OSError as e:
    raise RuntimeError(f'Network/download error loading VAE: {e}.') from e
print('✅ VAE loaded.')

print('\nLoading ControlNets (MLSD + Depth)...')
try:
    controlnet_mlsd = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-mlsd',
        torch_dtype=torch.float16,
        cache_dir=cache_dir,
    )
    controlnet_depth = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-depth',
        torch_dtype=torch.float16,
        cache_dir=cache_dir,
    )
except OSError as e:
    raise RuntimeError(f'Network/download error loading ControlNet: {e}.') from e
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError('CUDA OOM loading ControlNet. Try Runtime -> Factory reset runtime.') from e
print('✅ Both ControlNets loaded.')

print('\nLoading Realistic Vision V6.0 + MultiControlNet pipeline...')
try:
    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        'SG161222/Realistic_Vision_V6.0_B1_noVAE',
        controlnet=[controlnet_mlsd, controlnet_depth],
        vae=vae,
        torch_dtype=torch.float16,
        safety_checker=None,
        cache_dir=cache_dir,
    ).to('cuda')
except OSError as e:
    raise RuntimeError(f'Network/download error loading Realistic Vision: {e}.') from e
except ImportError as e:
    raise RuntimeError(f'Missing dependency: {e}. Re-run Section 3.') from e
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError('CUDA OOM loading pipeline. Try Runtime -> Factory reset runtime.') from e

print('\n✅ Pipeline loaded: Realistic Vision V6.0 + MultiControlNet(MLSD+Depth) + sd-vae-ft-mse')


## Section 7b — Scheduler Swap (DPM++ 2M Karras) — sec 5.2

DPM++ Karras at 22 steps produces equal or better image quality than PNDM at 30 steps.
This partially offsets the added cost of the second ControlNet.


In [ ]:
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='dpmsolver++',
)
print('✅ Scheduler switched to DPM++ 2M Karras.')
print(f'   NUM_INFERENCE_STEPS = {NUM_INFERENCE_STEPS}  (22 steps; was 30 with PNDM)')


## Section 8 — Memory Optimization & Prompt Truncation Setup


In [ ]:
try:
    pipe.enable_xformers_memory_efficient_attention()
    print('✅ xformers memory-efficient attention enabled.')
except Exception as e:
    print(f'⚠️  xformers not available ({e}). Using standard attention.')

pipe.enable_vae_slicing()
print('✅ VAE slicing enabled.')
pipe.enable_attention_slicing()
print('✅ Attention slicing enabled.')

allocated = torch.cuda.memory_allocated() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\nVRAM after model load: {allocated:.2f} / {total_mem:.1f} GB')


def truncate_to_clip(text, max_tokens=77):
    # Truncate text to SD1.5 CLIP 77-token limit using the pipeline tokenizer.
    # Pre-truncating suppresses the tokenizer warning AND guarantees important
    # early tokens (style adjectives, furniture names) are always kept.
    ids = pipe.tokenizer(
        text,
        truncation=True,
        max_length=max_tokens,
        return_tensors='pt',
    ).input_ids[0]
    return pipe.tokenizer.decode(ids, skip_special_tokens=True)


print('\n✅ truncate_to_clip() helper defined.')


## Section 9 — Helpers: Image Preprocessing & Letterbox


In [ ]:
def preprocess_image(image, size=None):
    # Convert to RGB and resize to target resolution (default 512x512).
    if size is None:
        size = IMG_SIZE
    return image.convert('RGB').resize(size, Image.LANCZOS)


def letterbox_to_square(image, size=512, fill=(128, 128, 128)):
    # Resize image to fit within size x size while preserving aspect ratio.
    # Pads the shorter dimension with fill colour.
    # Returns (padded_square_image, crop_box) where crop_box removes the padding.
    # Example: 1920x1080 -> scaled to 512x288, padded to 512x512.
    image = image.convert('RGB')
    w, h  = image.size
    scale = size / max(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    resized = image.resize((new_w, new_h), Image.LANCZOS)

    canvas = Image.new('RGB', (size, size), fill)
    left   = (size - new_w) // 2
    top    = (size - new_h) // 2
    canvas.paste(resized, (left, top))

    crop_box = (left, top, left + new_w, top + new_h)
    return canvas, crop_box


def remove_letterbox(image, crop_box, target_size):
    # Crop letterbox padding and resize back to target_size (original resolution).
    return image.crop(crop_box).resize(target_size, Image.LANCZOS)


print('✅ preprocess_image(), letterbox_to_square(), remove_letterbox() defined.')


## Section 10 — Helper: Room Classification (CLIP Zero-Shot)


In [ ]:
print('Loading CLIP model (ViT-Large-Patch14)...')
clip_model = CLIPModel.from_pretrained(
    'openai/clip-vit-large-patch14',
    cache_dir=cache_dir,
).to('cuda')
clip_processor = CLIPProcessor.from_pretrained(
    'openai/clip-vit-large-patch14',
    cache_dir=cache_dir,
)
print('✅ CLIP (ViT-L/14) loaded.')


def classify_room(image):
    # Zero-shot room classification using prompt ensembling.
    # For each class, similarity averaged across all templates in CLIP_LABEL_PROMPTS[class].
    # Returns (room_type, confidence).
    labels      = list(CLIP_LABEL_PROMPTS.keys())
    class_scores = []
    img_inputs  = clip_processor(images=image, return_tensors='pt').to('cuda')

    with torch.no_grad():
        image_features = clip_model.get_image_features(**img_inputs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        for label in labels:
            templates   = CLIP_LABEL_PROMPTS[label]
            text_inputs = clip_processor(
                text=templates, return_tensors='pt', padding=True
            ).to('cuda')
            text_features = clip_model.get_text_features(**text_inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            sims = (image_features @ text_features.T).squeeze(0)
            class_scores.append(sims.mean().item())

    scores_tensor = torch.tensor(class_scores)
    probs    = torch.softmax(scores_tensor * 100.0, dim=0)
    best_idx = int(probs.argmax().item())
    return labels[best_idx], float(probs[best_idx])


print('✅ classify_room() defined (ViT-L/14 + prompt ensembling).')


## Section 10b — Optional: Measure Real Classification Accuracy

Create `/content/labeled_test/` with subfolders `bedroom`, `living_room`, `kitchen`, `office`, `dining_room`
and drop real photos into each. Skip entirely if you don't have labeled photos — this is optional.


In [ ]:
TEST_DIR = '/content/labeled_test'

if not os.path.isdir(TEST_DIR):
    print(f'No labeled test folder at {TEST_DIR} -- skipping (optional).')
else:
    correct, total, per_class = 0, 0, {}
    for true_label in ROOM_LABELS:
        class_dir = os.path.join(TEST_DIR, true_label)
        if not os.path.isdir(class_dir):
            continue
        files = [f for f in os.listdir(class_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        class_correct = 0
        for fname in files:
            img = Image.open(os.path.join(class_dir, fname)).convert('RGB')
            pred_label, conf = classify_room(img)
            total  += 1
            is_ok   = (pred_label == true_label)
            correct += int(is_ok)
            class_correct += int(is_ok)
            mark = '✅' if is_ok else '❌'
            print(f'  {mark} true={true_label:12s} pred={pred_label:12s} conf={conf:.1%}  ({fname})')
        if files:
            per_class[true_label] = class_correct / len(files)

    if total > 0:
        print(f'\n{"="*50}')
        print(f'Overall accuracy: {correct}/{total} = {correct/total:.1%}')
        for cls, acc in per_class.items():
            print(f'  {cls:14s}: {acc:.1%}')
        print(f'{"="*50}')
    else:
        print('No labeled images found in class subfolders.')


## Section 11 — Helper: Structure Extraction (MLSD)


In [ ]:
print('Loading MLSD detector...')
mlsd_detector = MLSDdetector.from_pretrained('lllyasviel/ControlNet', cache_dir=cache_dir)
print('✅ MLSD detector loaded.')


def extract_structure(image):
    # Extract MLSD structural line map (walls/windows/doors) from a room photo.
    return mlsd_detector(image)


print('✅ extract_structure() defined.')


## Section 11b — Helper: Depth Extraction (MiDaS) — sec 5.3

MiDaS encodes continuous near/far geometry across the whole image. Combined with MLSD
(which gives structural lines but no depth), this grounds furniture placement and tells
the diffusion model where the floor plane is — the primary fix for floating furniture.

Depth runs directly on the 512px working copy (smooth signal; no quality benefit from upscaling).


In [ ]:
print('Loading MiDaS depth detector...')
try:
    midas_detector = MidasDetector.from_pretrained('lllyasviel/ControlNet', cache_dir=cache_dir)
    print('✅ MiDaS depth detector loaded.')
except Exception as e:
    raise RuntimeError(f'Failed to load MiDaS detector: {e}.') from e


def extract_depth(image):
    # Extract MiDaS depth map -- grounds furniture placement and near/far scale.
    # Runs on the letterboxed 512px working copy (smooth signal; MLSD benefits
    # from a 1024px upscale but depth does not -- saves ~1-2s per request).
    return midas_detector(image)


print('✅ extract_depth() defined.')


## Section 12 — Helper: Prompt Builder

**Verbatim port from `backend/app/services/style_templates.py` and `backend/app/services/prompt_builder.py`.**  
Do NOT paraphrase or improve the prompt text — identical (room_type, style) pairs must produce
byte-for-byte identical prompts in the backend and this notebook.


In [ ]:
# Verbatim port of backend/app/services/style_templates.py
STYLE_TEMPLATES = {
    'modern_minimalist': {
        'furniture': [
            'Low-profile platform sofa with hidden supports',
            'Monolithic concrete or stone coffee table',
            'Handleless matte-finish storage cabinets',
            'Recessed track lighting with slim spotlights',
        ],
        'palette': ['Pure White', 'Warm Charcoal', 'Soft Beige', 'Matte Black'],
        'budget_tag': 'Mid-Range',
        'reason_template': (
            'Focuses on clean lines, simplicity, and functionality by eliminating unnecessary '
            'clutter, creating a calm and open atmosphere in your {room_type}.'
        ),
    },
    'scandinavian': {
        'furniture': [
            'Light oak tapered-leg side tables',
            'Woven wool lounge chair in cream or soft beige',
            'Floating wood shelves with concealed mounting',
            'Minimalist opal glass pendant lighting',
        ],
        'palette': ['Alabaster White', 'Pale Ash Oak', 'Mist Grey', 'Sage Green accents'],
        'budget_tag': 'Budget-Friendly',
        'reason_template': (
            'Emphasizes natural light, organic textures, and functional simplicity. '
            'Warm wood tones introduce hygge into your {room_type}.'
        ),
    },
    'industrial': {
        'furniture': [
            'Distressed leather clean-lined sofa',
            'Reclaimed wood and steel coffee table',
            'Black iron piping wall shelving units',
            'Exposed filament Edison bulb light fixtures',
        ],
        'palette': ['Raw Iron Black', 'Brick Red', 'Weathered Grey', 'Copper accents'],
        'budget_tag': 'Mid-Range',
        'reason_template': (
            'Celebrates raw materials and architectural features, giving your {room_type} '
            'a bold warehouse-like aesthetic.'
        ),
    },
    'bohemian': {
        'furniture': [
            'Rattan or wicker peacock accent chair',
            'Plush low-to-the-ground floor pillows and poufs',
            'Layered distressed vintage Persian rugs',
            'Macrame wall hangings and hanging plant holders',
        ],
        'palette': ['Warm Terracotta', 'Mustard Yellow', 'Olive Green', 'Burnt Orange'],
        'budget_tag': 'Budget-Friendly',
        'reason_template': (
            'Embraces a carefree, layered, eclectically styled environment with rich textiles '
            'and vibrant plant life in your {room_type}.'
        ),
    },
    'luxury_contemporary': {
        'furniture': [
            'Velvet upholstered custom sofa with brass plinth',
            'Polished Calacatta marble console table',
            'Bespoke geometric crystal chandelier',
            'Polished gold or brass metal frame accent chairs',
        ],
        'palette': ['Champagne Gold', 'Rich Emerald', 'Pearl White', 'Deep Obsidian'],
        'budget_tag': 'Premium',
        'reason_template': (
            'Merges high-end materials and sophisticated textures, delivering an elegant '
            'luxury aesthetic in your {room_type}.'
        ),
    },
}


# Verbatim port of backend/app/services/prompt_builder.py
# sec 5.4 -- appended to every negative_prompt, not replacing it.
# Root cause of floating furniture is architectural (depth ControlNet); these reinforce it.
NEGATIVE_PROMPT_SUFFIX = (
    ', furniture not touching floor, objects without shadows, '
    'disconnected objects, inconsistent scale, floating rugs, '
    'melted textures, warped carpet patterns'
)


def build_prompt(room_type, style):
    # Build (prompt, negative_prompt). Byte-for-byte identical to backend prompt_builder.py.
    style_key = style.lower().strip().replace(' ', '_')
    room_name = room_type.lower().strip().replace('_', ' ')

    if style_key not in STYLE_TEMPLATES:
        style_key = 'modern_minimalist'
    template = STYLE_TEMPLATES[style_key]

    palette_str   = ', '.join(template['palette'])
    furniture_str = ', '.join(template['furniture'])

    prompt = (
        f'A professionally designed {room_name} in {style_key.replace(chr(95), " ").title()} style. '
        f'Featuring: {furniture_str}. '
        f'Color palette: {palette_str}. '
        f'Interior design photography, realistic shadows, high-end materials, 8k resolution, '
        f'architectural digest, daytime soft lighting, clean composition, same room perspective, '
        f'same camera angle, same wall positions, same window positions, same door positions, '
        f'same ceiling height, same room proportions, same floor plan.'
    )

    negative_prompt = (
        'low quality, blurry, worst quality, deformed, distorted, unrealistic proportions, '
        'cluttered, messy, extra walls, missing windows, out of frame, watermark, text, signature'
        + NEGATIVE_PROMPT_SUFFIX
    )

    return prompt, negative_prompt


# Sanity check
_tp, _tn = build_prompt('living_room', 'scandinavian')
assert 'Scandinavian' in _tp or 'scandinavian' in _tp, 'Style missing from prompt'
assert 'furniture not touching floor' in _tn, 'NEGATIVE_PROMPT_SUFFIX missing'
print('✅ STYLE_TEMPLATES and build_prompt() defined and verified.')
print(f'   Positive sample: {_tp[:100]}...')
print(f'   Negative suffix: ...{_tn[-80:]}')


## Section 13 — Core Inference Function


In [ ]:
def generate_variations(
    original_image,
    control_image_mlsd,
    control_image_depth,
    prompt,
    negative_prompt,
):
    # Produce len(SEEDS) variations using MultiControlNet (MLSD + Depth).
    # MLSD: preserves wall/window/door geometry.
    # Depth: grounds furniture placement and near/far scale.
    # Together they fix floating furniture without sacrificing geometry preservation.
    #
    # Per-conditioning weights [1.0, 0.55]: MLSD slightly dominant for geometry,
    # depth secondary for grounding. Tune these two numbers first if results look off.
    #
    # Batched-with-sequential-fallback contract is UNCHANGED.
    # SEEDS = [42, 123, 777] must not be altered.
    # Returns (variations, elapsed_seconds) where each variation is
    # {'image': PIL.Image, 'seed': int}.
    start = time.time()

    prompt_safe = truncate_to_clip(prompt)
    neg_safe    = truncate_to_clip(negative_prompt)

    control_images      = [control_image_mlsd, control_image_depth]
    conditioning_scales = [1.0, 0.55]

    variations = []
    try:
        generators = [torch.Generator('cuda').manual_seed(s) for s in SEEDS]
        result = pipe(
            prompt=[prompt_safe] * len(SEEDS),
            negative_prompt=[neg_safe] * len(SEEDS),
            image=original_image,
            control_image=[control_images] * len(SEEDS),
            strength=STRENGTH,
            guidance_scale=GUIDANCE_SCALE,
            controlnet_conditioning_scale=conditioning_scales,
            num_inference_steps=NUM_INFERENCE_STEPS,
            num_images_per_prompt=1,
            generator=generators,
        )
        if len(result.images) != len(SEEDS):
            raise RuntimeError(
                f'Batched call returned {len(result.images)} images, expected {len(SEEDS)}.'
            )
        variations = [
            {'image': img, 'seed': seed}
            for img, seed in zip(result.images, SEEDS)
        ]
    except Exception as batch_exc:
        print(f'⚠️  Batched generation failed ({batch_exc}). Falling back to sequential.')
        variations = []
        for seed in SEEDS:
            torch.cuda.empty_cache()
            gen = torch.Generator('cuda').manual_seed(seed)
            single = pipe(
                prompt=prompt_safe,
                negative_prompt=neg_safe,
                image=original_image,
                control_image=control_images,
                strength=STRENGTH,
                guidance_scale=GUIDANCE_SCALE,
                controlnet_conditioning_scale=conditioning_scales,
                num_inference_steps=NUM_INFERENCE_STEPS,
                num_images_per_prompt=1,
                generator=gen,
            )
            variations.append({'image': single.images[0], 'seed': seed})

    return variations, time.time() - start


print('✅ generate_variations() defined (MultiControlNet: MLSD+Depth, batched+sequential fallback).')


## Section 14 — Metadata & Timing Assembly


In [ ]:
def pil_to_b64(img):
    # Encode PIL image to base64 PNG string.
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def assemble_response(
    room_type, confidence, style, prompt,
    control_image, variations, gen_time, total_time,
    control_depth=None,
):
    # Build the JSON response schema required by the backend integration contract.
    # control_depth_b64 is additive-optional: None when not provided.
    return {
        'room_type':           room_type,
        'room_confidence':     round(confidence, 4),
        'style':               style,
        'prompt_used':         prompt,
        'model_used':          'realistic-vision-v6+multicontrolnet-mlsd-depth',
        'control_image_b64':   pil_to_b64(control_image),
        'control_depth_b64':   pil_to_b64(control_depth) if control_depth is not None else None,
        'generation_time_sec': round(gen_time, 2),
        'total_time_sec':      round(total_time, 2),
        'variations': [
            {'seed': v['seed'], 'image_b64': pil_to_b64(v['image'])}
            for v in variations
        ],
    }


print('✅ pil_to_b64() and assemble_response() defined.')


## Section 15 — Visualization (Sanity-Check Plot)

Run this offline sanity check **before** starting the server to catch OOM or model errors early.  
MLSD may return a near-blank map on the synthetic test image (expected). What matters: inference completes without OOM.


In [ ]:
gradient = np.zeros((512, 512, 3), dtype=np.uint8)
gradient[:, :, 0] = np.linspace(60,  200, 512, dtype=np.uint8)
gradient[:, :, 2] = np.linspace(200,  60, 512, dtype=np.uint8)
gradient[150:350, 100:400, :] = [200, 200, 200]
gradient[300:500, 150:350, :] = [140, 110, 80]
sanity_pil = preprocess_image(Image.fromarray(gradient))
print('Running sanity check (modern_minimalist, MultiControlNet)...')

try:
    ctrl_img_mlsd  = extract_structure(sanity_pil)
    ctrl_img_depth = extract_depth(sanity_pil)

    ctrl_arr = np.array(ctrl_img_mlsd)
    if ctrl_arr.max() < 5:
        print('⚠️  MLSD blank -- normal for synthetic images; depth ControlNet still active.')

    room_t, conf    = classify_room(sanity_pil)
    prompt_s, neg_s = build_prompt(room_t, 'modern_minimalist')
    vars_s, elapsed = generate_variations(
        sanity_pil, ctrl_img_mlsd, ctrl_img_depth, prompt_s, neg_s
    )

    fig = plt.figure(figsize=(18, 5))
    fig.suptitle(
        f'Sanity Check -- {room_t} ({conf:.1%}) | modern_minimalist | MultiControlNet | {elapsed:.1f}s',
        fontsize=12
    )
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax0 = fig.add_subplot(gs[0]); ax0.imshow(sanity_pil);    ax0.set_title('Original'); ax0.axis('off')
    ax1 = fig.add_subplot(gs[1]); ax1.imshow(ctrl_img_mlsd); ax1.set_title('MLSD');    ax1.axis('off')
    for idx_v, v in enumerate(vars_s):
        ax = fig.add_subplot(gs[2 + idx_v])
        ax.imshow(v['image']); ax.set_title(f'seed={v["seed"]}'); ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/sanity_check_output.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Sanity check passed in {elapsed:.2f}s.')
    if conf < 0.5:
        print(f'   Note: {conf:.1%} confidence on synthetic image is expected.')

except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    print('OOM during sanity check. Runtime -> Factory reset runtime, re-run from Section 2.')
except Exception as e:
    print(f'Sanity check failed: {e}')
    raise


## Section 16 — FastAPI App Definition


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger('roomcanvas')

app = FastAPI(
    title='RoomCanvas Inference Server',
    description=(
        'Realistic Vision V6.0 + MultiControlNet (MLSD + MiDaS Depth) '
        'inference endpoint for RoomCanvas AI.'
    ),
    version='2.0.0',
)

print('✅ FastAPI app defined.')


## Section 17 — Health Check Endpoint


In [ ]:
@app.get('/health')
def health():
    # Return server status and GPU stats.
    # Polled by the startup cell -- MUST return HTTP 200 before ngrok is created.
    return {
        'status': 'ok',
        'model':  'realistic-vision-v6+multicontrolnet-mlsd-depth',
        'gpu': (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
        ),
        'vram_allocated_mb': (
            round(torch.cuda.memory_allocated() / 1024**2, 1)
            if torch.cuda.is_available() else 0
        ),
        'vram_total_mb': (
            round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 1)
            if torch.cuda.is_available() else 0
        ),
    }


print('✅ GET /health defined.')


## Section 18 — Main /infer Endpoint


In [ ]:
@app.post('/infer')
async def infer(image: UploadFile = File(...), style: str = Form(...)):
    # Full pipeline:
    #   image validation -> MLSD + depth extraction -> room classification
    #   -> prompt building -> batched 3-variation generation -> response.
    #
    # Dimension contract:
    #   - Uploaded image letterboxed to 512x512 (aspect-ratio-preserving, neutral-grey padding).
    #   - MLSD extraction: 1024x1024 upscale, then downscaled back to 512 (line detection benefits).
    #   - Depth extraction: 512px working copy (smooth signal, no upscale benefit, saves ~1-2s).
    #   - Diffusion: 512px working copy.
    #   - After generation: padding cropped, content upscaled to original resolution.
    #   - Returned images always match original upload dimensions. No squash/stretch.
    #
    # Response schema:
    #   room_type, room_confidence, style, prompt_used, model_used,
    #   control_image_b64, control_depth_b64 (additive, may be null),
    #   generation_time_sec, total_time_sec,
    #   variations: [{seed, image_b64}, ...] (3 items, seeds 42/123/777)
    request_start = time.time()

    if style not in STYLE_TEMPLATES:
        raise HTTPException(
            status_code=400,
            detail=f'Unknown style "{style}". Valid: {list(STYLE_TEMPLATES.keys())}',
        )

    try:
        contents     = await image.read()
        pil_original = Image.open(io.BytesIO(contents)).convert('RGB')
        original_width, original_height = pil_original.size
    except Exception as exc:
        logger.error(f'Image load failed: {exc}')
        raise HTTPException(status_code=400, detail=f'Invalid image file: {exc}')

    pil_working, crop_box = letterbox_to_square(pil_original, size=IMG_SIZE[0])

    # Stage 1: structure + depth extraction
    t0 = time.time()
    try:
        mlsd_input    = pil_original.resize((1024, 1024), Image.LANCZOS)
        control_mlsd  = extract_structure(mlsd_input).resize(IMG_SIZE, Image.LANCZOS)
        control_depth = extract_depth(pil_working)
    except Exception as exc:
        logger.error(f'Structure/depth extraction failed: {exc}')
        raise HTTPException(status_code=500, detail='Structure extraction failed')
    logger.info(f'Structure+depth: {round(time.time()-t0, 2)}s')

    # Stage 2: room classification
    t0 = time.time()
    try:
        room_type, confidence = classify_room(pil_working)
    except Exception as exc:
        logger.error(f'CLIP classification failed: {exc}')
        raise HTTPException(status_code=500, detail='Room classification failed')
    logger.info(f'CLIP: {round(time.time()-t0, 2)}s  {room_type} ({confidence:.1%})')

    # Stage 3: prompt building
    prompt, negative_prompt = build_prompt(room_type, style)

    # Stage 4: batched MultiControlNet generation (one OOM retry)
    try:
        variations, gen_time = generate_variations(
            pil_working, control_mlsd, control_depth, prompt, negative_prompt
        )
    except torch.cuda.OutOfMemoryError:
        logger.warning('CUDA OOM -- clearing cache and retrying once')
        torch.cuda.empty_cache()
        try:
            variations, gen_time = generate_variations(
                pil_working, control_mlsd, control_depth, prompt, negative_prompt
            )
        except torch.cuda.OutOfMemoryError:
            logger.error('CUDA OOM on retry')
            raise HTTPException(
                status_code=500,
                detail='Out of GPU memory. Try again in a few seconds.'
            )
    logger.info(f'Generation: {gen_time:.2f}s')

    # Stage 5: crop letterbox + upscale back to original resolution
    original_size = (original_width, original_height)
    for v in variations:
        v['image'] = remove_letterbox(v['image'], crop_box, original_size)

    total_time = round(time.time() - request_start, 2)
    logger.info(f'Total request: {total_time}s')

    return assemble_response(
        room_type=room_type,
        confidence=confidence,
        style=style,
        prompt=prompt,
        control_image=control_mlsd,
        variations=variations,
        gen_time=round(gen_time, 2),
        total_time=total_time,
        control_depth=control_depth,
    )


print('✅ POST /infer defined (MultiControlNet: MLSD+Depth, letterbox+crop pipeline).')


## Section 19 — Server Startup (Background)

> Starts uvicorn in a daemon thread using the Python 3.12-safe `uvicorn.Server` API,
> then polls `/health` until the server accepts connections.
> Only after `200 OK` does execution continue to the ngrok cell.


In [ ]:
nest_asyncio.apply()

# Python 3.12 no longer auto-creates an event loop in worker threads.
# Fix: use uvicorn.Server API + create a fresh event loop inside the thread.
config = uvicorn.Config(
    app,
    host='0.0.0.0',
    port=8000,
    loop='asyncio',  # never use uvloop inside a non-main thread
    reload=False,    # reload=True spawns processes, breaks in Colab
    workers=1,
    access_log=False,
    log_level='warning',
)
server = uvicorn.Server(config)


def run_server():
    # Thread target: creates its own event loop (required for Python 3.12).
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(server.serve())
    finally:
        loop.close()


print('🚀 Starting uvicorn (Python 3.12-safe background thread)...')
server_thread = threading.Thread(target=run_server, daemon=True, name='uvicorn-server')
server_thread.start()

print('⏳ Waiting for http://127.0.0.1:8000/health ...')
server_up = False
for attempt in range(30):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            server_up = True
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        pass
    time.sleep(1)

if server_up:
    print('✅ Server reachable on http://127.0.0.1:8000')
    print('   Run Section 20 to open the ngrok tunnel.')
else:
    print('❌ Server did NOT come up in 30s. Thread alive:', server_thread.is_alive())
    raise RuntimeError(
        'FastAPI/uvicorn failed to start on port 8000. '
        'Check that no other process owns port 8000 and re-run this cell.'
    )


## Section 20 — Ngrok Tunnel Setup

**Get your free ngrok auth token:**
1. Sign up at [ngrok.com](https://ngrok.com/) (free, no credit card)
2. Go to **Your Authtoken** in the ngrok dashboard
3. Paste the token into Section 5's `NGROK_AUTH_TOKEN` field, then re-run Section 5

The public URL **changes every restart** — paste the new URL into `backend/.env` as `COLAB_INFERENCE_URL`.


In [ ]:
if not NGROK_AUTH_TOKEN:
    raise ValueError(
        'NGROK_AUTH_TOKEN is empty. Paste your free token into Section 5 and re-run Section 5.'
    )

print('🌐 Creating ngrok tunnel...')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect('127.0.0.1:8000')
public_url = tunnel.public_url

print(f'\n✅ Public endpoint ready!')
print(f'   Health: {public_url}/health')
print(f'   Infer:  POST {public_url}/infer')
print(f'\n📋 Paste into backend/.env as COLAB_INFERENCE_URL: {public_url}')


## Section 21 — Live Test Cell

**Run after Section 20.** Tests all 5 valid styles, one invalid style (expect HTTP 400),
and a corrupt upload (expect HTTP 400).


In [ ]:
from google.colab import files as colab_files

test_img_path = '/content/test_room.jpg'
if not os.path.exists(test_img_path):
    print('Upload a room photo (JPG/PNG):')
    uploaded = colab_files.upload()
    if uploaded:
        test_img_path = list(uploaded.keys())[0]
    else:
        print('No file uploaded -- using neutral grey stand-in.')
        Image.fromarray(np.full((512, 512, 3), 180, dtype=np.uint8)).save(test_img_path)

STYLES_TO_TEST = [
    'modern_minimalist', 'scandinavian', 'industrial', 'bohemian', 'luxury_contemporary'
]
SERVER_URL = str(public_url)

test_results = []
print(f'Testing {len(STYLES_TO_TEST)} styles against {SERVER_URL}/infer...\n')

for style_name in STYLES_TO_TEST:
    print(f'  -> {style_name} ... ', end='', flush=True)
    try:
        with open(test_img_path, 'rb') as fh:
            resp = requests.post(
                f'{SERVER_URL}/infer',
                files={'image': ('room.jpg', fh, 'image/jpeg')},
                data={'style': style_name},
                timeout=120,
            )
        if resp.status_code == 200:
            data = resp.json()
            test_results.append((style_name, data))
            print(
                f'OK  room={data["room_type"]} ({data["room_confidence"]:.1%}) '
                f'total={data["total_time_sec"]}s gen={data["generation_time_sec"]}s'
            )
        else:
            print(f'FAIL HTTP {resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        print(f'ERROR: {e}')

for style_name, data in test_results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f'{style_name} | {data["room_type"]} ({data["room_confidence"]:.1%}) | {data["total_time_sec"]}s',
        fontsize=11
    )
    for ax, v in zip(axes, data['variations']):
        pil_v = Image.open(io.BytesIO(base64.b64decode(v['image_b64'])))
        ax.imshow(pil_v); ax.set_title(f'Seed {v["seed"]}'); ax.axis('off')
    plt.tight_layout()
    plt.show()

print('\nTesting invalid style key (expect HTTP 400)...')
with open(test_img_path, 'rb') as fh:
    bad = requests.post(
        f'{SERVER_URL}/infer',
        files={'image': ('r.jpg', fh, 'image/jpeg')},
        data={'style': 'not_valid'},
        timeout=10,
    )
assert bad.status_code == 400, f'Expected 400 got {bad.status_code}'
print(f'OK HTTP 400: {bad.json()}')

print('Testing corrupt file upload (expect HTTP 400)...')
bad2 = requests.post(
    f'{SERVER_URL}/infer',
    files={'image': ('bad.jpg', b'not an image', 'image/jpeg')},
    data={'style': 'scandinavian'},
    timeout=10,
)
assert bad2.status_code == 400, f'Expected 400 got {bad2.status_code}'
print(f'OK HTTP 400: {bad2.json()}')

print(f'\n✅ Live test complete. {len(test_results)}/{len(STYLES_TO_TEST)} styles passed.')


## Section 22 — Performance Diagnostics


In [ ]:
vram_alloc = torch.cuda.memory_allocated() / 1024**3
vram_res   = torch.cuda.memory_reserved()  / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('-' * 52)
print('  VRAM Diagnostics')
print('-' * 52)
print(f'  Allocated : {vram_alloc:.2f} GB')
print(f'  Reserved  : {vram_res:.2f} GB')
print(f'  Total     : {vram_total:.1f} GB')
print(f'  Free      : {vram_total - vram_res:.2f} GB')

if test_results:
    total_times = [d['total_time_sec']      for _, d in test_results]
    gen_times   = [d['generation_time_sec'] for _, d in test_results]
    avg_total   = sum(total_times) / len(total_times)
    avg_gen     = sum(gen_times)   / len(gen_times)

    print()
    print('-' * 52)
    print('  Timing Breakdown (from live test)')
    print('-' * 52)
    for sn, d in test_results:
        print(f'  {sn:25s}  total={d["total_time_sec"]:5.1f}s  gen={d["generation_time_sec"]:5.1f}s')
    print(f'  {"AVERAGE":25s}  total={avg_total:5.1f}s  gen={avg_gen:5.1f}s')

    THRESHOLD = 60
    print()
    if avg_total <= THRESHOLD:
        print(f'  ✅ Average total: {avg_total:.1f}s -- within demo range (<{THRESHOLD}s)')
    else:
        print(f'  ⚠️  Average total: {avg_total:.1f}s -- EXCEEDS {THRESHOLD}s.')
        print('      Reduce NUM_INFERENCE_STEPS to 18-20 in Section 5.')
else:
    print('\n  ⚠️  No test_results -- run Section 21 first.')


## Section 23 — Outputs, Export & Download

Saves all generated images, pipeline metadata, and inference metrics to `outputs/`, then zips for download.  
**Run after Section 21 has completed successfully.**


In [ ]:
import zipfile
from datetime import datetime

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

DIRS = ['/content/outputs/images', '/content/outputs/reports', '/content/outputs/model']
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('✅ Output directories created.')

saved_images = []
if 'test_results' in dir() and test_results:
    for style_name, data in test_results:
        style_dir = f'/content/outputs/images/{style_name}'
        os.makedirs(style_dir, exist_ok=True)
        for v in data['variations']:
            img   = Image.open(io.BytesIO(base64.b64decode(v['image_b64'])))
            fname = f'{style_dir}/seed_{v["seed"]}.png'
            img.save(fname, dpi=(300, 300))
            saved_images.append(fname)
    print(f'✅ Saved {len(saved_images)} images to outputs/images/')
else:
    print('⚠️  test_results not found -- run Section 21 first.')

model_info_lines = [
    'RoomCanvas AI -- Model & Pipeline Information',
    f'Generated: {TIMESTAMP}',
    '',
    '=== Inference Pipeline ===',
    'Stable Diffusion:        SG161222/Realistic_Vision_V6.0_B1_noVAE',
    'VAE:                     stabilityai/sd-vae-ft-mse',
    'ControlNet MLSD:         lllyasviel/sd-controlnet-mlsd',
    'ControlNet Depth:        lllyasviel/sd-controlnet-depth',
    'CLIP Classifier:         openai/clip-vit-large-patch14',
    'Detector MLSD:           lllyasviel/ControlNet (MLSDdetector)',
    'Detector Depth:          lllyasviel/ControlNet (MidasDetector)',
    'Scheduler:               DPM++ 2M Karras',
    '',
    '=== Generation Parameters ===',
    f'Inference steps:         {NUM_INFERENCE_STEPS}',
    f'Strength:                {STRENGTH}',
    f'Guidance scale:          {GUIDANCE_SCALE}',
    'ControlNet scales:       [1.0, 0.55]  (MLSD, Depth)',
    f'Seeds:                   {SEEDS}',
    'Variations per request:  3',
    '',
    '=== Room Classification ===',
    f'Classes:                 {ROOM_LABELS}',
    'CLIP templates/class:    4 (ensemble averaged)',
    '',
    '=== Styles Supported ===',
    'modern_minimalist, scandinavian, industrial, bohemian, luxury_contemporary',
]
with open('/content/outputs/model/model_info.txt', 'w') as fh:
    fh.write('\n'.join(model_info_lines))
print('✅ Saved outputs/model/model_info.txt')

metrics = {
    'timestamp':                     TIMESTAMP,
    'pipeline':                      'realistic-vision-v6+multicontrolnet-mlsd-depth',
    'scheduler':                     'DPM++ 2M Karras',
    'strength':                      STRENGTH,
    'guidance_scale':                GUIDANCE_SCALE,
    'controlnet_conditioning_scale': [1.0, 0.55],
    'num_inference_steps':           NUM_INFERENCE_STEPS,
    'seeds':                         SEEDS,
    'img_size':                      list(IMG_SIZE),
}
if 'test_results' in dir() and test_results:
    tts   = [d['total_time_sec']      for _, d in test_results]
    gts   = [d['generation_time_sec'] for _, d in test_results]
    confs = [d['room_confidence']     for _, d in test_results]
    metrics.update({
        'avg_total_time_sec':      round(sum(tts)   / len(tts),   2),
        'avg_generation_time_sec': round(sum(gts)   / len(gts),   2),
        'min_total_time_sec':      round(min(tts),                2),
        'max_total_time_sec':      round(max(tts),                2),
        'styles_tested':           [s for s, _ in test_results],
        'avg_clip_confidence':     round(sum(confs) / len(confs), 4),
    })
with open('/content/outputs/reports/inference_metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print('✅ Saved outputs/reports/inference_metrics.json')

if 'test_results' in dir() and test_results:
    lines = [
        'RoomCanvas AI -- Inference Report',
        f'Generated: {TIMESTAMP}',
        '=' * 60,
        f'{"Style":<25} {"Room Type":<15} {"Confidence":>11} {"Total(s)":>9} {"Gen(s)":>7}',
        '-' * 60,
    ]
    for sn, d in test_results:
        lines.append(
            f'{sn:<25} {d["room_type"]:<15} {d["room_confidence"]:>10.1%} '
            f'{d["total_time_sec"]:>9.1f} {d["generation_time_sec"]:>7.1f}'
        )
    lines += [
        '=' * 60,
        f'Avg CLIP confidence: {metrics.get("avg_clip_confidence", 0):.1%}',
        f'Avg total time:      {metrics.get("avg_total_time_sec", 0):.1f}s',
        f'Avg generation time: {metrics.get("avg_generation_time_sec", 0):.1f}s',
    ]
    with open('/content/outputs/reports/classification_report.txt', 'w') as fh:
        fh.write('\n'.join(lines))
    print('✅ Saved outputs/reports/classification_report.txt')

zip_path = f'/content/RoomCanvas_Output_{TIMESTAMP}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('/content/outputs'):
        for fname in fnames:
            full    = os.path.join(root, fname)
            arcname = os.path.relpath(full, '/content')
            zf.write(full, arcname)
print(f'✅ Created {zip_path}')

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print('✅ Download triggered.')
except Exception as e:
    print(f'⚠️  Auto-download failed ({e}). Download manually from: {zip_path}')


## Section 24 — Cleanup (Optional)

Run only when done. Restart the runtime before starting a new session.


In [ ]:
try:
    ngrok.disconnect(public_url)
    print('✅ ngrok tunnel disconnected.')
except Exception as e:
    print(f'⚠️  ngrok disconnect: {e}')

try:
    del pipe, controlnet_mlsd, controlnet_depth, vae, clip_model, mlsd_detector, midas_detector
    torch.cuda.empty_cache()
    print('✅ Models deleted and CUDA cache cleared.')
except Exception as e:
    print(f'⚠️  Cleanup: {e}')

print('\nResources released. Restart runtime before next session for a clean state.')


## Section 25 — Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `CUDA OOM` at model load | T4 not selected, or another notebook using VRAM | Runtime → Factory reset runtime → re-run from Section 2 |
| `No GPU detected` at Section 2 | Wrong runtime type | Runtime → Change runtime type → T4 GPU → Save |
| `pip install failed` | Transient Colab network error | Re-run Section 3 |
| MLSD map is blank | Normal for low-texture or synthetic images | Expected; depth ControlNet still grounds furniture |
| Floating furniture persists | `conditioning_scales` needs tuning | In `generate_variations()` try `[0.9, 0.7]` (raise depth first) |
| Flat grey walls | Depth scale too high | In `generate_variations()` try `[1.0, 0.4]` (lower depth) |
| ngrok URL not working | Session restarted | Re-run Section 20, paste new URL into `backend/.env` |
| Server not up after 30s | Port conflict or uvicorn crash | Check for another process on 8000, re-run Section 19 |
| `/infer` returns 400 on valid style | Style key typo | Valid: `modern_minimalist`, `scandinavian`, `industrial`, `bohemian`, `luxury_contemporary` |
| Generation time >60s | Dual-ControlNet overhead | Reduce `NUM_INFERENCE_STEPS` to 18-20 in Section 5 |
